# ⚙️ Notebook 02: Pré-processamento e Preparação dos Dados
**Projeto**: Explainable Movie Recommender  
**Modulo**: `src/preprocessing.py`  

---

## 🎯 Objetivos deste Notebook
1. **Limpeza e Tratamento de Nulos**: Tratar dados ausentes e garantir tipos numéricos consistentes.
2. **Filtragem Iterativa k-Core**: Garantir que cada usuário possua no mínimo 20 avaliações e cada filme tenha no mínimo 10 avaliações.
3. **Split Treino/Teste (80/20) Estratificado por Usuário**: Garantir a distribuição proporcional de treino e teste para cada usuário individualmente.
4. **Persistência**: Verificar os arquivos serializados em `data/processed/`.

In [1]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np

ROOT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

from src.config import PROCESSED_DATA_DIR, SEED
from src.preprocessing import run_preprocessing_pipeline

print("Ambiente de pré-processamento configurado!")

Ambiente de pré-processamento configurado!


## 1. Execução da Pipeline de Pré-processamento

In [2]:
train_df, test_df, movies_df, users_df = run_preprocessing_pipeline()

=== INICIANDO PIPELINE DE PRÉ-PROCESSAMENTO ===
Dados brutos carregados: 100,000 avaliações.
[Limpeza] Preenchidos valores ausentes em 'release_date' dos filmes.
[Filtragem k-core] Concluída em 3 iterações. Registros finais: 97,737 (usuários >= 20, filmes >= 10).
[Split Treino/Teste] Treino: 77,812 linhas (79.6%) | Teste: 19,925 linhas (20.4%)
[Persistência] Todos os arquivos salvos com sucesso em: D:\Codes\CineMatch\data\processed
=== PIPELINE DE PRÉ-PROCESSAMENTO CONCLUÍDA COM SUCESSO! ===


## 2. Validação da Integridade do Split e dos Limiares

In [3]:
print("=== VALIDAÇÃO DO SPLIT POR USUÁRIO ===")
full_df = pd.concat([train_df, test_df])
user_train_counts = train_df.groupby('user_id').size()
user_test_counts = test_df.groupby('user_id').size()
user_full_counts = full_df.groupby('user_id').size()

print(f"Total de usuários únicos no treino: {train_df['user_id'].nunique()}")
print(f"Total de usuários únicos no teste : {test_df['user_id'].nunique()}")
print(f"Menor número de avaliações por usuário no conjunto total: {user_full_counts.min()}")

movie_full_counts = full_df.groupby('item_id').size()
print(f"Menor número de avaliações por filme no conjunto total  : {movie_full_counts.min()}")

print("\n=== AMOSTRA DO CONJUNTO DE TREINO ===")
print(train_df.head())

print("\n=== AMOSTRA DO CONJUNTO DE TESTE ===")
print(test_df.head())

=== VALIDAÇÃO DO SPLIT POR USUÁRIO ===
Total de usuários únicos no treino: 932
Total de usuários únicos no teste : 932
Menor número de avaliações por usuário no conjunto total: 20
Menor número de avaliações por filme no conjunto total  : 10

=== AMOSTRA DO CONJUNTO DE TREINO ===
 user_id  item_id  rating  timestamp            datetime
     420      603     4.0  891356864 1998-03-31 15:07:44
     150      121     2.0  878747322 1997-11-05 16:28:42
     294      603     5.0  889854323 1998-03-14 05:45:23
     566      496     5.0  881649428 1997-12-09 06:37:08
     314      399     3.0  877889359 1997-10-26 18:09:19

=== AMOSTRA DO CONJUNTO DE TESTE ===
 user_id  item_id  rating  timestamp            datetime
     806      210     5.0  882387520 1997-12-17 19:38:40
     443      271     4.0  883504682 1997-12-30 17:58:02
     708      751     4.0  892718687 1998-04-16 09:24:47
     225      492     4.0  879539767 1997-11-14 20:36:07
     284      319     3.0  885329238 1998-01-20 20:47:1

## 3. Inspeção dos Arquivos Gerados em `data/processed/`

In [4]:
files = list(PROCESSED_DATA_DIR.glob("*.csv"))
print("=== ARQUIVOS GERADOS EM DATA/PROCESSED/ ===")
for f in sorted(files):
    size_mb = f.stat().st_size / (1024 * 1024)
    print(f"- {f.name:<25} | Tamanho: {size_mb:.2f} MB")

=== ARQUIVOS GERADOS EM DATA/PROCESSED/ ===
- processed_movies.csv      | Tamanho: 0.16 MB
- processed_ratings.csv     | Tamanho: 3.99 MB
- processed_users.csv       | Tamanho: 0.02 MB
- test_ratings.csv          | Tamanho: 0.81 MB
- train_ratings.csv         | Tamanho: 3.17 MB


## 4. Conclusão da Etapa 2

O conjunto de dados está devidamente limpo, filtrado com o algoritmo *k-core*, dividido em Treino/Teste estratificado por usuário e persistido em `data/processed/`.

---
**Próxima Etapa no Roadmap**: **Etapa 3 — Treinamento dos Modelos de Recomendação** (`src/train.py` e `notebooks/03_treinamento.ipynb`).